# 🔗 LangChain Setup & Simple Chain — LCEL Version

**What this notebook does:** the exact same "tell me a joke" chain as `Langchain_Setup_and_Simple_Chain- DONE.ipynb`, rewritten using LCEL's pipe (`|`) syntax instead of the older `LLMChain` class — same behavior, same model, same prompt, just a different (more modern) style. See *LangChain.md* Q2 for what each part of this style actually means.

### 📦 Cell 0 — Install the libraries

Same as the original, minus one package: `langchain-classic` is no longer needed, since this version doesn't use `LLMChain` at all.

In [ ]:
!pip install langchain cohere langchain_community langchain-cohere

### 🧰 Cell 1 — Import the building blocks

Same `PromptTemplate` and `ChatCohere` as before. The one new import is `StrOutputParser` — an LCEL chain needs an explicit **parser** as its last pipe step to turn the model's raw response into a plain string. The old `LLMChain` did that conversion for you automatically behind the scenes, which is exactly why the original notebook had to dig the answer out with `response['text']`.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_cohere import ChatCohere
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

### 🔗 Cell 2 — Build the three pieces, and pipe them together

This is the actual conversion. Instead of wrapping everything inside one `LLMChain(...)` object, each piece is built on its own, then wired together with `|`:

- **`prompt`** — the same dynamic template as before.
- **`model`** — the same `ChatCohere` model, just no longer passed *into* a chain class — it's now a pipeline stage in its own right.
- **`parser`** — new: `StrOutputParser()` takes the model's raw response and extracts just the plain text.

**Example:** `prompt | model | parser` reads as *“fill in the prompt, send it to the model, then hand the model's answer to the parser to clean it up”* — the exact same three jobs the old `LLMChain` was doing internally, just spelled out explicitly instead of hidden inside one class.

In [ ]:
prompt = PromptTemplate(
    input_variables=["adjective"],
    template="Tell me a {adjective} joke"
)

model = ChatCohere(cohere_api_key=userdata.get('COHERE_KEY'), model="command-a-03-2025")

parser = StrOutputParser()

chain = prompt | model | parser

### ▶️ Cell 3 — Run the chain

Same idea as `.invoke("funny")` in the original, but LCEL chains expect their input as a **dictionary** matching the template's variable name — so it's `chain.invoke({"adjective": "funny"})` instead of a bare string.

In [ ]:
result = chain.invoke({"adjective": "funny"})

### 🖨️ Cell 4 — Print the result

No need to dig into a dictionary this time — `StrOutputParser` already handed back a plain string, so `result` *is* the joke itself.

In [ ]:
print(result)

### 📝 What actually changed, side by side

| | Old (`LLMChain`) | New (LCEL) |
|---|---|---|
| Build the chain | `LLMChain(llm=model, prompt=prompt, verbose=True)` | `chain = prompt \| model \| parser` |
| Run it | `llm.invoke("funny")` | `chain.invoke({"adjective": "funny"})` |
| Get the answer | `response['text']` | `result` — already a plain string |
| Seeing intermediate steps | `verbose=True` built in | Not available directly on the chain — LCEL relies on LangChain's tracing/callbacks (e.g. LangSmith) instead |

Same end result, same model, same prompt — just expressed as an explicit pipe of independent pieces instead of one bundled class. Worth noting honestly: this isn't a strict feature-for-feature upgrade — `verbose=True` was a genuinely convenient one-line debugging switch that LCEL doesn't replicate directly.